In [ ]:
import traceback
import sys
import os
import time
import asyncio
import csv
import datetime

today_date = datetime.date.today()

from datetime import datetime, timedelta
from NetDrivers import NetDevice


# Define the file name for the CSV
csv_filename = "LN2_all_fields.csv" #str(today_date) + "_LN2_level.csv"

ln = NetDevice(server_ip="192.168.0.201",server_port=7180)

#ln2_level = ln._sendCmd("FILL:B?")


start_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
end_time = start_time + timedelta(hours=24)

with open(csv_filename, "w", newline='') as csv_file:
    csv_writer = csv.writer(csv_file)

    # Header of the CSV:
    csv_writer.writerow(["Current Time", "LN2 reader Date", "LN2 reader Time", "N2 level (current units)","N2 level measurement period (micro-s)", "Sys Local Control lock", "Display N2/He","Level Units (%,cm,in)","Level B % (lower limit when to start auto-refill)", "Level A % (upper limit)"]) 
    
    #end_of_day = datetime.now() + timedelta(hours=24)
    #end_time = end_of_day - datetime.now()
    # Data in a file for a day
    while datetime.now() < end_time:
        time_now = datetime.now()
        current_time = time_now.time()
        
        ln2_date = ln._sendCmd("SYStem:DATE?")
        ln2_time = ln._sendCmd("SYStem:TIME?")
        ln2_level_curr = ln._sendCmd("MEASure:N2:LEVel?",getResponse=True,verbose=False)
        ln2_period = ln._sendCmd("MEASure:N2:PERIod?")
        ln2_lock_st = ln._sendCmd("SYStem:KLOCK?")
        ln2_disp = ln._sendCmd("DISPLAY:N2?")
        ln2_units = ln._sendCmd("N2:UNIT?")
        ln2_levelB = ln._sendCmd("FILL:B?")
        ln2_level_strB = ln2_levelB[0]
        ln2_levelA = ln._sendCmd("FILL:A?")
        ln2_level_strA = ln2_levelA[0]
        
        unix_timestamp = time_now.timestamp() # Convert it to a Unix timestamp
        csv_writer.writerow([current_time.strftime("%H:%M:%S"), ln2_date, ln2_time,ln2_level_curr,ln2_period,ln2_lock_st, ln2_disp,ln2_units, str(ln2_level_strB), str(ln2_level_strA)])
        csv_file.flush()
        time.sleep(10)
        
#print("Time = ",ln._sendCmd("SYStem:TIME?"))